# Olist star schema on Microsoft Fabric (PySpark)

Builds `fact_sales`, `dim_customers`, `dim_products` and `dim_date` from the staging tables loaded by Dataflow Gen2 (`stg_orders`, `stg_order_items`, `stg_customers`, `stg_products`, `stg_category_translation`) in the Lakehouse `lh_olist`. Attach `lh_olist` as the default lakehouse and run the cells top to bottom.

## 1. Fact table and customer dimension

In [ ]:
%%pyspark
from pyspark.sql import functions as F

orders = (spark.table("stg_orders")
          .select("order_id", "customer_id", "order_status",
                  F.to_date("order_purchase_timestamp").alias("order_date"),
                  "delivery_days", "is_late"))
items = (spark.table("stg_order_items")
         .select("order_id", "order_item_id", "product_id", "seller_id",
                 F.col("price").cast("double").alias("price"),
                 F.col("freight_value").cast("double").alias("freight_value")))

# Fact table: one row per order item
fact_sales = items.join(orders, "order_id", "inner")
fact_sales.write.mode("overwrite").format("delta").saveAsTable("fact_sales")

# Customer dimension
dim_customers = (spark.table("stg_customers")
                 .select("customer_id", "customer_unique_id", "customer_city", "customer_state")
                 .dropDuplicates(["customer_id"]))
dim_customers.write.mode("overwrite").format("delta").saveAsTable("dim_customers")

## 2. Product and date dimensions

`stg_category_translation` arrived with columns `Column1` and `Column2` (headers were not promoted in the dataflow), so they are renamed here before the join.

In [1]:
%%pyspark
from pyspark.sql import functions as F

# Fix headers of the translation table
trans = (spark.table("stg_category_translation")
         .withColumnRenamed("Column1", "product_category_name")
         .withColumnRenamed("Column2", "product_category_name_english")
         .filter(F.col("product_category_name") != "product_category_name"))

# Product dimension (blank categories become "unknown")
prod = spark.table("stg_products").withColumn(
    "product_category_name",
    F.when(F.trim(F.col("product_category_name")) == "", None)
     .otherwise(F.col("product_category_name")))
dim_products = (prod.join(trans, "product_category_name", "left")
    .select("product_id",
            F.coalesce("product_category_name_english",
                       "product_category_name", F.lit("unknown")).alias("category"),
            "product_weight_g"))
dim_products.write.mode("overwrite").format("delta").saveAsTable("dim_products")

# Date dimension
fact_sales = spark.table("fact_sales")
b = fact_sales.agg(F.min("order_date"), F.max("order_date")).first()
dates = spark.sql(f"SELECT explode(sequence(to_date('{b[0]}'), to_date('{b[1]}'), interval 1 day)) AS date")
dim_date = dates.select("date", F.year("date").alias("year"), F.quarter("date").alias("quarter"),
                        F.month("date").alias("month_num"),
                        F.date_format("date", "MMM").alias("month_name"),
                        F.date_format("date", "EEEE").alias("weekday"))
dim_date.write.mode("overwrite").format("delta").saveAsTable("dim_date")

## 3. Row count check

In [2]:
%%pyspark
for t in ["fact_sales", "dim_customers", "dim_products", "dim_date"]:
    print(t, spark.table(t).count())

fact_sales 112650
dim_customers 99441
dim_products 32951
dim_date 730


## 4. Last order date

September 2018 is a partial month, so trend charts in the report stop at August 2018.

In [3]:
%%sql
SELECT MAX(order_date) FROM fact_sales

2018-09-03